In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import plotnine as p9
import liana as li
import muon as mu
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
from liana.method._pipe_utils._common import _get_props
from liana.method.sp._utils import _add_complexes_to_var
from liana._logging import _logg
from liana.method._pipe_utils import prep_check_adata, assert_covered
from scipy.sparse import csr_matrix
from pathlib import Path
from plotnine import scale_y_continuous
from plotnine import geom_vline, annotate


In [3]:
merfish_path = Path("/g/stegle/aalsayah/repos/data/merfish")

In [4]:
adata_main = sc.read(merfish_path / "mouse_brain/WB_MERFISH_animal2_coronal.h5ad")

In [5]:
adata = adata_main[adata_main.obs["brain_section_label"] == "C57BL6J-2.034"]

# Basic QC

In [6]:
# filter cells and genes
sc.pp.filter_cells(adata, min_genes=10)
sc.pp.filter_genes(adata, min_cells=3)

/home/alsayah/.local/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:176: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


In [7]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Main function: 1-nearest neighbors between every pair of cell types

In [8]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

def compute_pairwise_nearest_neighbor_df(cell_types, coordinates, ratio=None):
    """
    Compute 1-nearest neighbors between every pair of cell types.

    For each cell of type A, find its nearest neighbor among cells of type B,
    for all A–B combinations (including A=B).

    Parameters
    ----------
    cell_types : array-like of shape (n_cells,)
        Cell type labels (e.g. adata.obs["cell_type"]).
    coordinates : array-like of shape (n_cells, n_dims)
        Spatial coordinates of each cell (e.g. adata.obsm["spatial"]).
        Units are assumed to be µm unless `ratio` is provided.
    ratio : float, optional
        Conversion factor from pixels to micrometers (µm per pixel).
        If provided, coordinates are multiplied by this ratio.

    Returns
    -------
    pairwise_df : pandas.DataFrame
        Columns:
        ["cell_type_1", "cell_index", "cell_type_2",
         "nearest_neighbor_index", "distance_um"]
    """
    cell_types = np.asarray(cell_types)
    coordinates = np.asarray(coordinates, dtype=float)
    n_cells = coordinates.shape[0]

    # Handle unit conversion
    if ratio is not None:
        if ratio <= 0:
            raise ValueError("`ratio` must be positive (µm per pixel).")
        print(f"Converting coordinates from pixels to µm using ratio = {ratio:.3f}")
        coordinates = coordinates * ratio
    else:
        print("Assuming coordinates are already in micrometers (µm).")

    unique_types = np.unique(cell_types)
    results = []

    # Loop over all type pairs (A,B)
    for type_a in unique_types:
        idx_a = np.where(cell_types == type_a)[0]
        coords_a = coordinates[idx_a]

        for type_b in unique_types:
            idx_b = np.where(cell_types == type_b)[0]
            coords_b = coordinates[idx_b]

            if len(idx_a) == 0 or len(idx_b) == 0:
                continue

            # If A == B and only one cell, skip (no neighbor)
            if type_a == type_b and len(idx_b) == 1:
                continue

            # Build NN model on type B
            k = 2 if type_a == type_b else 1
            nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
            nn.fit(coords_b)
            distances, indices = nn.kneighbors(coords_a)

            # Exclude self for A==B
            if type_a == type_b:
                distances = distances[:, 1]
                indices = indices[:, 1]
            else:
                distances = distances[:, 0]
                indices = indices[:, 0]

            # Map indices in B back to global cell indices
            nn_indices_global = idx_b[indices]

            # Store results
            df_tmp = pd.DataFrame({
                "cell_type_1": type_a,
                "cell_index": idx_a,
                "cell_type_2": type_b,
                "nearest_neighbor_index": nn_indices_global,
                "distance_um": distances,
            })
            results.append(df_tmp)

    pairwise_df = pd.concat(results, ignore_index=True)
    return pairwise_df


In [9]:
pairwise_df = compute_pairwise_nearest_neighbor_df(
    adata.obs["cell_type"],
    adata.obsm["X_spatial_coords"])

pairwise_df

Assuming coordinates are already in micrometers (µm).


,cell_type_1,cell_index,cell_type_2,nearest_neighbor_index,distance_um
0,GABAergic neuron,75,GABAergic neuron,251,65.882696
1,GABAergic neuron,96,GABAergic neuron,175,37.848759
2,GABAergic neuron,145,GABAergic neuron,152,76.686703
3,GABAergic neuron,152,GABAergic neuron,145,76.686703
4,GABAergic neuron,170,GABAergic neuron,215,47.134626
...,...,...,...,...,...
982215,vascular leptomeningeal cell,49044,vascular leptomeningeal cell,49048,34.009090
982216,vascular leptomeningeal cell,49048,vascular leptomeningeal cell,49058,4.837012
982217,vascular leptomeningeal cell,49058,vascular leptomeningeal cell,49048,4.837012
982218,vascular leptomeningeal cell,49063,vascular leptomeningeal cell,49038,11.746987


In [10]:
pairwise_df.sort_values("cell_index")

,cell_type_1,cell_index,cell_type_2,nearest_neighbor_index,distance_um
967305,vascular leptomeningeal cell,0,astrocyte,8,13.781606
971230,vascular leptomeningeal cell,0,ependymal cell,24657,2951.294151
975155,vascular leptomeningeal cell,0,microglial cell,22,20.266370
972015,vascular leptomeningeal cell,0,glutamatergic neuron,3,80.041886
975940,vascular leptomeningeal cell,0,monocyte,983,619.772141
...,...,...,...,...,...
889905,oligodendrocyte,49110,pericyte,47157,70.135842
866631,oligodendrocyte,49110,neuroblast (sensu Vertebrata),47741,1186.914394
897663,oligodendrocyte,49110,smooth muscle cell,45033,493.660723
882147,oligodendrocyte,49110,oligodendrocyte precursor cell,47088,148.807977


In [11]:
# Output length should be =  (n_cell_type x total cells)
print(adata.shape[0])
print(len(pairwise_df), len(adata.obs["cell_type"].unique()))
print(adata.shape[0] * len(adata.obs["cell_type"].unique()))

49111
982220 20
982220
